# Lab 5: Deep Learning & LLMs for NLP

**Course:** Natural Language Processing


**Objectives:**
- Understand RNN, LSTM, GRU architectures for sequence modeling
- Use pre-trained Transformers for NER
- Interact with LLMs via API for text generation

---

## Instructions

1. Complete all exercises marked with `# YOUR CODE HERE`
2. **Answer all written questions** in the designated markdown cells
3. Save your completed notebook
4. **Push to your Git repository and send the link to: yoroba93@gmail.com**

---

## Lab Structure

| Part | Model | Task |
|------|-------|------|
| A | RNN | Character-level Language Model |
| B | LSTM | Sentiment Analysis |
| C | GRU | News Classification |
| D | Transformer | Named Entity Recognition | 
| E | LLM (Mistral) | Text Generation & QA |

---

## Setup

In [2]:
# Install required libraries (uncomment if needed)
# !pip install torch transformers datasets requests numpy pandas matplotlib

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cpu
PyTorch version: 2.13.0


---

# PART A: RNN - Character-Level Language Model (10 min)

**Use Case:** Predict the next character for autocomplete.

**Dataset:** Tiny Shakespeare

In [4]:
# Load Tiny Shakespeare text
# HF dataset "tiny_shakespeare" breaks with newer huggingface_hub versions
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
with urllib.request.urlopen(url) as resp:
    text = resp.read().decode("utf-8")[:10000]  # first 10K chars for speed

print(f"Text length: {len(text)} characters")
print(f"Sample: {text[:200]}")

Text length: 10000 characters
Sample: First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [5]:
# Create character vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {''.join(chars[:30])}...")

Vocabulary size: 57
Characters: 
 !',-.:;?ABCDEFHIJLMNOPRSTUVW...


In [6]:
# Prepare sequences
seq_length = 30
X, y = [], []

for i in range(len(text) - seq_length):
    X.append([char_to_idx[c] for c in text[i:i+seq_length]])
    y.append(char_to_idx[text[i+seq_length]])

X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

print(f"Sequences: {X.shape[0]}, Sequence length: {seq_length}")

Sequences: 9970, Sequence length: 30


In [7]:
# Simple RNN model
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])  # Last timestep
        return out

# Create model
rnn_model = CharRNN(vocab_size, embed_dim=32, hidden_dim=64).to(device)
print(f"RNN Parameters: {sum(p.numel() for p in rnn_model.parameters()):,}")

RNN Parameters: 11,801


### Exercise A.1: Train the RNN

In [8]:
# Hyperparameters
batch_size = 128
epochs = 5
learning_rate = 0.005  # tried 0.001 first but loss dropped faster with 0.005

# DataLoader
dataset = TensorDataset(X[:5000], y[:5000])  # subset so it trains fast on CPU
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(rnn_model.parameters(), lr=learning_rate)

# Standard PyTorch training loop
losses = []
for epoch in range(epochs):
    total_loss = 0
    for batch_X, batch_y in loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        output = rnn_model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 1/5, Loss: 3.0721
Epoch 2/5, Loss: 2.5451
Epoch 3/5, Loss: 2.3273
Epoch 4/5, Loss: 2.2029
Epoch 5/5, Loss: 2.1099


In [9]:
# Generate text
def generate_text(model, start_str, length=100):
    model.eval()
    chars_generated = list(start_str)
    input_seq = [char_to_idx.get(c, 0) for c in start_str[-seq_length:]]
    
    with torch.no_grad():
        for _ in range(length):
            x = torch.tensor([input_seq[-seq_length:]]).to(device)
            output = model(x)
            pred_idx = torch.argmax(output, dim=1).item()
            chars_generated.append(idx_to_char[pred_idx])
            input_seq.append(pred_idx)
    
    return ''.join(chars_generated)

# Test generation
print("Generated text:")
print(generate_text(rnn_model, "To be or not", length=100))

Generated text:
To be or not Citizen:
Were the the the the the the the the the the the the the the the the the the the the the t


---

# PART B: LSTM - Sentiment Analysis 

**Use Case:** Classify movie review sentiment.

**Dataset:** IMDB Reviews

In [10]:
# Load IMDB dataset (streaming avoids disk/cache issues)
from datasets import load_dataset

train_texts, train_labels = [], []
for i, row in enumerate(load_dataset("stanfordnlp/imdb", split="train", streaming=True)):
    if i >= 1000:
        break
    train_texts.append(row["text"])
    train_labels.append(row["label"])

test_texts, test_labels = [], []
for i, row in enumerate(load_dataset("stanfordnlp/imdb", split="test", streaming=True)):
    if i >= 200:
        break
    test_texts.append(row["text"])
    test_labels.append(row["label"])

print(f"Train: {len(train_texts)}, Test: {len(test_texts)}")

Train: 1000, Test: 200


In [11]:
# Simple tokenization and vocabulary
from collections import Counter
import re

def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())[:100]  # Max 100 tokens

# Build vocabulary from training data
all_tokens = [tok for text in train_texts for tok in tokenize(text)]
vocab = {word: idx+2 for idx, (word, _) in enumerate(Counter(all_tokens).most_common(5000))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

print(f"Vocabulary size: {len(vocab)}")

Vocabulary size: 5002


In [12]:
# Encode texts
def encode_text(text, max_len=100):
    tokens = tokenize(text)
    encoded = [vocab.get(t, 1) for t in tokens]  # 1 = UNK
    padded = encoded[:max_len] + [0] * (max_len - len(encoded))
    return padded[:max_len]

X_train = torch.tensor([encode_text(t) for t in train_texts])
y_train = torch.tensor(train_labels)
X_test = torch.tensor([encode_text(t) for t in test_texts])
y_test = torch.tensor(test_labels)

print(f"Train shape: {X_train.shape}")

Train shape: torch.Size([1000, 100])


### Exercise B.1: Complete the LSTM Model

In [13]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # LSTM layer: reads the sequence and keeps a memory of past tokens
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.embedding(x)
        x = self.dropout(x)

        # hidden shape: (num_layers, batch, hidden_dim) - we take the last layer
        lstm_out, (hidden, cell) = self.lstm(x)
        out = self.fc(hidden[-1])
        return out

lstm_model = LSTMClassifier(
    vocab_size=len(vocab),
    embed_dim=64,
    hidden_dim=64,
    num_classes=2
).to(device)

print(f"LSTM Parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")

LSTM Parameters: 353,538


In [14]:
# Quick training
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)

# Train for 3 epochs
for epoch in range(3):
    lstm_model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        output = lstm_model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# Evaluate
lstm_model.eval()
with torch.no_grad():
    test_output = lstm_model(X_test.to(device))
    preds = torch.argmax(test_output, dim=1).cpu()
    acc = (preds == y_test).float().mean()
    print(f"\nTest Accuracy: {acc:.4f}")

Epoch 1, Loss: 0.4198
Epoch 2, Loss: 0.0035
Epoch 3, Loss: 0.0005

Test Accuracy: 1.0000


---

# PART C: GRU - News Classification

**Use Case:** Classify news articles by topic.

**Why GRU?** Fewer parameters than LSTM, faster training.

In [15]:
# Load AG News (updated dataset name for newer HF hub)
from datasets import load_dataset

ag_news = load_dataset("sh0416/ag_news")
ag_train = ag_news['train'].shuffle(seed=42).select(range(2000))
ag_test = ag_news['test'].shuffle(seed=42).select(range(500))

ag_labels = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}
print(f"Classes: {list(ag_labels.values())}")

Classes: ['World', 'Sports', 'Business', 'Sci/Tech']


### Exercise C.1: Build GRU Classifier

In [19]:
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # GRU is like LSTM but simpler (2 gates instead of 3)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)

        # GRU returns hidden state only, no separate cell state
        gru_out, hidden = self.gru(x)
        out = self.fc(hidden[-1])
        return out

ag_tokens = [tok for item in ag_train for tok in tokenize(item['description'])]
ag_vocab = {word: idx+2 for idx, (word, _) in enumerate(Counter(ag_tokens).most_common(5000))}
ag_vocab['<PAD>'] = 0
ag_vocab['<UNK>'] = 1

def encode_ag(text, vocab, max_len=50):
    tokens = tokenize(text)
    encoded = [vocab.get(t, 1) for t in tokens]
    return (encoded[:max_len] + [0] * max_len)[:max_len]

X_ag_train = torch.tensor([encode_ag(item['description'], ag_vocab) for item in ag_train])
y_ag_train = torch.tensor([item['label'] - 1 for item in ag_train])  # dataset uses 1-4, model needs 0-3
X_ag_test = torch.tensor([encode_ag(item['description'], ag_vocab) for item in ag_test])
y_ag_test = torch.tensor([item['label'] - 1 for item in ag_test])

print(f"AG News - Train: {X_ag_train.shape}, Test: {X_ag_test.shape}")

AG News - Train: torch.Size([2000, 50]), Test: torch.Size([500, 50])


In [22]:
gru_model = GRUClassifier(
    vocab_size=len(ag_vocab),
    embed_dim=64,
    hidden_dim=64,
    num_classes=4
).to(device)

print(f"GRU Parameters: {sum(p.numel() for p in gru_model.parameters()):,}")
print(f"(Compare to LSTM: GRU has fewer parameters!)")

# Make sure labels are 0-3 (this dataset uses 1-4)
if y_ag_train.min() >= 1:
    y_ag_train = y_ag_train - 1
    y_ag_test = y_ag_test - 1
print(f"Label range: {y_ag_train.min().item()} to {y_ag_train.max().item()}")

# Quick training on AG News
ag_loader = DataLoader(TensorDataset(X_ag_train, y_ag_train), batch_size=32, shuffle=True)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(gru_model.parameters(), lr=0.001)

for epoch in range(3):
    gru_model.train()
    total_loss = 0
    for batch_X, batch_y in ag_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        loss = criterion(gru_model(batch_X), batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"GRU Epoch {epoch+1}, Loss: {total_loss/len(ag_loader):.4f}")

gru_model.eval()
with torch.no_grad():
    preds = torch.argmax(gru_model(X_ag_test.to(device)), dim=1).cpu()
    acc = (preds == y_ag_test).float().mean()
    print(f"GRU Test Accuracy: {acc:.4f}")

GRU Parameters: 345,348
(Compare to LSTM: GRU has fewer parameters!)


IndexError: Target 4 is out of bounds.

---

# PART D: Transformer - Named Entity Recognition

**Use Case:** Extract entities from text.

**Dataset:** CoNLL-2003

In [ ]:
# Use pre-trained NER model from Hugging Face
from transformers import pipeline

# Load NER pipeline (uses BERT-based model)
print("Loading NER model...")
ner_pipeline = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")
print("Model loaded!")

In [ ]:
# Example NER
text = "Apple Inc. was founded by Steve Jobs in Cupertino, California. Tim Cook is the current CEO."

entities = ner_pipeline(text)
print(f"Text: {text}\n")
print("Entities found:")
for ent in entities:
    print(f"  {ent['word']:20} -> {ent['entity_group']:10} (score: {ent['score']:.3f})")

### Exercise D.1: NER on Your Own Texts

In [ ]:
# 3 sentences with people, organizations, and locations
my_sentences = [
    "Barack Obama gave a speech at the United Nations in New York about climate change.",
    "Elon Musk announced that Tesla will open a new factory in Berlin next year.",
    "My friend Sarah works at Google in London and travels to Paris every month.",
]

for sent in my_sentences:
    if sent != "___":
        print(f"\nText: {sent}")
        entities = ner_pipeline(sent)
        for ent in entities:
            print(f"  {ent['word']:20} -> {ent['entity_group']}")

---

# PART E: LLM - Text Generation with Mistral API

**Use Case:** Conversational AI and Question Answering.

**Setup:** Get a free API key from https://console.mistral.ai/

In [ ]:
import os

# Put your key in an env variable so it doesn't end up on GitHub
# Get a free key at: https://console.mistral.ai/
MISTRAL_API_KEY = os.environ.get("MISTRAL_API_KEY", "___")

In [ ]:
import requests

def query_mistral(prompt, max_tokens=150):
    """Query Mistral API."""
    url = "https://api.mistral.ai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {MISTRAL_API_KEY}",
        "Content-Type": "application/json"
    }
    data = {
        "model": "mistral-small-latest",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens
    }
    
    response = requests.post(url, headers=headers, json=data)
    if response.status_code == 200:
        return response.json()['choices'][0]['message']['content']
    else:
        return f"Error: {response.status_code} - {response.text}"

# Test (only if API key is set)
if MISTRAL_API_KEY != "___":
    response = query_mistral("What is NLP in one sentence?")
    print(f"Mistral: {response}")
else:
    print("Please set your MISTRAL_API_KEY above.")

### Exercise E.1: Compare LLM with Traditional Models

In [ ]:
# TODO: Ask Mistral to perform sentiment analysis and compare with our LSTM

test_review = "This movie was absolutely terrible. The acting was bad and the plot made no sense."

# LLM approach
if MISTRAL_API_KEY != "___":
    prompt = f"""Classify the sentiment of this review as 'positive' or 'negative'. 
Just respond with one word.

Review: {test_review}

Sentiment:"""
    
    llm_result = query_mistral(prompt, max_tokens=10)
    print(f"LLM Sentiment: {llm_result}")

# Traditional LSTM approach (if model trained)
try:
    encoded = torch.tensor([encode_text(test_review)]).to(device)
    lstm_model.eval()
    with torch.no_grad():
        lstm_pred = torch.argmax(lstm_model(encoded)).item()
    print(f"LSTM Sentiment: {'positive' if lstm_pred == 1 else 'negative'}")
except:
    print("LSTM model not available")

In [ ]:
# TODO: Use LLM for summarization (something traditional models can't easily do)

long_text = """
Natural language processing (NLP) is a subfield of linguistics, computer science, 
and artificial intelligence concerned with the interactions between computers and 
human language, in particular how to program computers to process and analyze large 
amounts of natural language data. The result is a computer capable of understanding 
the contents of documents, including the contextual nuances of the language within them.
"""

if MISTRAL_API_KEY != "___":
    summary_prompt = f"Summarize this in one sentence:\n\n{long_text}"
    summary = query_mistral(summary_prompt, max_tokens=50)
    print(f"Summary: {summary}")

---

## Final Written Questions (Personal Interpretation)

Answer these questions based on YOUR experiments:

### Question 1: Model Architecture Comparison

Compare the parameter counts you observed:
- RNN: ~11,801 parameters
- LSTM: ~353,538 parameters
- GRU: ~345,348 parameters

**Why does LSTM have more parameters than GRU?** (Hint: think about gates)

**YOUR ANSWER:**

LSTM has more parameters than GRU because LSTM uses 3 gates (forget, input, output) plus a separate cell state. Each gate has its own weight matrices.

GRU only has 2 gates (reset and update) and combines the cell state and hidden state into one. Less gates = fewer weight matrices = fewer parameters. In my run GRU had about 8k fewer parameters than LSTM.

### Question 2: RNN vs LSTM for Long Sequences

**Why would LSTM perform better than vanilla RNN for sentiment analysis on long reviews?** Explain the vanishing gradient problem.

**YOUR ANSWER:**

A vanilla RNN only passes one hidden state through time. When you backpropagate through many steps, gradients get multiplied over and over and shrink (vanishing gradient). Early words in a long review barely affect the final prediction.

LSTM fixes this with a cell state and gates (forget, input, output). The cell state is like a highway where information can flow without getting crushed at each step. So the model can remember something from the start of a long review when classifying sentiment at the end.

### Question 3: Traditional Models vs LLMs

Based on your experiments:
1. **What can LLMs do that LSTM/GRU cannot?**
2. **What are the disadvantages of using LLM APIs?** (Think: cost, latency, privacy)
3. **When would you choose a traditional model over an LLM?**

**YOUR ANSWER:**

1. LLM advantages: summarization, open-ended QA, and zero-shot tasks without training a model from scratch. You just write a prompt. Our LSTM can only do binary sentiment on movie reviews.

2. LLM disadvantages: API costs money at scale, each call adds latency (network round trip), and you send data to a third party (privacy issue for sensitive text). Also needs internet.

3. When to use traditional models: simple classification on your own machine, when you have limited data and budget, when speed/latency matters (no API call), or when data must stay private (on-premise).

---

## Summary

| Model | Strength | Weakness | Best For |
|-------|----------|----------|----------|
| RNN | Simple, fast | Vanishing gradients | Short sequences |
| LSTM | Long-term memory | More parameters | Long text classification |
| GRU | Efficient, fast | Less expressive | When speed matters |
| Transformer | Parallel, contextual | Expensive | NER, QA, many tasks |
| LLM | Versatile, zero-shot | API cost, latency | Complex reasoning |

---

## Submission

- [ ] All code exercises completed
- [ ] All written questions answered
- [ ] Mistral API tested (or explained why not)
- [ ] **Push to Git and send link to: yoroba93@gmail.com**